# Hardware Benchmark — FPGA ZCU102

Publication-ready latency and energy figures for the manuscript (§5.4 Efficiency).

Two cuts:
1. **4-arch FPGA-only** (`s1`) — latency breakdown + energy/patch across all four architectures.
2. **3-platform** (`CPU / GPU / FPGA-s1`) — FP and ResSH only, compress scenario.

All data from `results/benchmark_hardware/` + `results/benchmark_unified/`.
Regenerate with `python scripts/benchmark/run_unified_benchmark.py --model-dir results/fpga/active_model/ --power`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
sys.path.insert(0, str(ROOT / "notebooks"))

from _benchmark_loader import load_runs, load_stage_breakdowns
from _plotkit import PALETTE, ARCH_LABEL, ARCH_ORDER as _ARCH_ORDER

FIG_DPI = 120
PLOTS_DIR = ROOT / "results" / "plots" / "hardware_benchmark"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MANUSCRIPT_DIR = ROOT / "LaTeX" / "SAR_DDC_FPGA_TGRS_2026" / "figures" / "images"
SAVE_FIGURES = True

PLAT = PALETTE["platforms"]
STAGE_CLR = PALETTE["cpp_stages"]

# Arch order for this notebook (FPGA-natural: FP → SH → ResFP → ResSH by complexity)
ARCH_ORDER = ["FP", "SHyp", "ResFP", "ResSHyp"]

# Stage stacking order
STAGE_ORDER = [
    "normalize",
    "g_a",
    "h_a",
    "eb_compress",
    "eb_decompress",
    "h_s",
    "gc_compress",
    "gc_decompress",
    "g_s",
    "denorm",
]
STAGE_DISPLAY = {"host_concat_abs": "concat_abs", "host_split_y_hat": "split_y_hat"}

# ── SERIES ────────────────────────────────────────────────────────────────────
SERIES = {
    "CPU": dict(
        platform="cpu", config="baseline", color=PLAT["cpu_dynamic"], hatch="", label="CPU"
    ),
    "GPU": dict(
        platform="gpu", config="baseline", color=PLAT["gpu_dynamic"], hatch="", label="GPU"
    ),
    "FPGA": dict(platform="fpga", config="s1", color=PLAT["fpga_dynamic"], hatch="", label="FPGA"),
}

# ── Load data ─────────────────────────────────────────────────────────────────
df = load_runs(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
sb = load_stage_breakdowns(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
df["edp_mJ_ms"] = df["energy_mJ_per_patch"] * df["total_latency_mean_ms"]


def _row(arch, spec, scenario):
    r = df[
        (df.arch == arch)
        & (df.platform == spec["platform"])
        & (df.config == spec["config"])
        & (df.scenario == scenario)
        & (df.model_name.str.contains("_L1000"))
    ]
    return r.iloc[0] if not r.empty else None


print(f"Loaded {len(df)} benchmark rows from {sorted(df.platform.unique())}")

## §1 · 4-arch FPGA-only latency breakdown

Per-stage latency breakdown for all four architectures on FPGA (s1, compress scenario).
Same as the F6 figure in `benchmark_cross_platform_analysis.ipynb` but FPGA-only and all 4 archs.

In [ ]:
# ── Stage styling for the latency breakdown ───────────────────────────────────
# stage_key → (legend_group_key, color, legend_label)
# Stages with the same legend_group_key share one legend entry but remain
# separate bar segments (heights are NOT summed).
# Colors reference STAGE_CLR (PALETTE["cpp_stages"]) from hw-01.
# For the two entropy groups, gc_compress / gc_decompress are the Okabe-Ito
# entries (vermillion #D55E00 and orange #E69F00); eb_ gets the same color.
STAGE_VIZ = {
    "normalize": ("normalize", STAGE_CLR["normalize"], "normalize"),
    "g_a": ("g_a", STAGE_CLR["g_a"], "$g_a$"),
    "h_a": ("h_a", STAGE_CLR["h_a"], "$h_a$"),
    "eb_compress": ("entropy_enc", STAGE_CLR["gc_compress"], "Entropy enc."),
    "gc_compress": ("entropy_enc", STAGE_CLR["gc_compress"], "Entropy enc."),
    "eb_decompress": ("entropy_dec", STAGE_CLR["gc_decompress"], "Entropy dec."),
    "gc_decompress": ("entropy_dec", STAGE_CLR["gc_decompress"], "Entropy dec."),
    "h_s": ("h_s", STAGE_CLR["h_s"], "$h_s$"),
    "g_s": ("g_s", STAGE_CLR["g_s"], "$g_s$"),
    "denorm": ("denorm", STAGE_CLR["denorm"], "denorm."),
}
LEGEND_GROUP_ORDER = [
    "normalize",
    "g_a",
    "h_a",
    "entropy_enc",
    "entropy_dec",
    "h_s",
    "g_s",
    "denorm",
]
_GRP_STYLE = {}
for _stg, (_grp, _clr, _lbl) in STAGE_VIZ.items():
    if _grp not in _GRP_STYLE:
        _GRP_STYLE[_grp] = (_clr, _lbl)


def stage_breakdown_fpga(
    archs=None,
    config="s1",
    scenario="compress",
    pct_labels=True,
    pct_min=0.06,
    save=None,
    manuscript_name=None,
):
    """Stacked per-stage latency, one bar per arch, FPGA only (s0 or s1).
    eb/gc entropy stages keep separate heights but share a color and legend entry."""
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    fig, ax = plt.subplots(figsize=(2.0 * len(archs) + 1.5, 5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.3, color="#aaa")
    seen_groups = []
    for xi, arch in enumerate(archs):
        spec = {"platform": "fpga", "config": config}
        r = _row(arch, spec, scenario)
        if r is None:
            continue
        st = sb[
            (sb.platform == "fpga")
            & (sb.model_name == r.model_name)
            & (sb.config == config)
            & (sb.scenario == scenario)
        ]
        total = st.mean_ms.sum()
        bottom = 0.0
        for stg in STAGE_ORDER:
            row_st = st[st.stage == stg]
            if row_st.empty:
                continue
            h = row_st.mean_ms.iloc[0]
            grp, clr, _ = STAGE_VIZ.get(stg, (stg, "#999", stg))
            ax.bar(
                xi, h, bottom=bottom, width=0.75, color=clr, edgecolor="white", lw=0.3, zorder=3
            )
            if pct_labels and total and h / total >= pct_min:
                ax.text(
                    xi,
                    bottom + h / 2,
                    f"{100 * h / total:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white",
                    fontweight="bold",
                )
            if grp not in seen_groups:
                seen_groups.append(grp)
            bottom += h
        if bottom:
            ax.text(xi, bottom, f"{bottom:.1f}", ha="center", va="bottom", fontsize=11)
    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel("Latency [ms]")
    handles = [
        Patch(facecolor=_GRP_STYLE[grp][0], label=_GRP_STYLE[grp][1])
        for grp in reversed(LEGEND_GROUP_ORDER)
        if grp in seen_groups
    ]
    ax.legend(
        handles=handles,
        loc="upper left",
        ncol=1,
        fontsize=10,
        framealpha=0.9,
    )
    fig.tight_layout()
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    if manuscript_name:
        dst = MANUSCRIPT_DIR / f"{manuscript_name}.pdf"
        fig.savefig(dst, bbox_inches="tight")
        print(f"→ manuscript: {dst}")
    plt.show()


# ── 4-arch FPGA-only (s1, compress) ──────────────────────────────────────────
stage_breakdown_fpga(
    save="fig_latency_fpga_4arch",
    manuscript_name="fig_latency_fpga_4arch",
)

In [ ]:
# ── Per-stage latency breakdown (absolute ms, full scenario, s0 + s1) ─────────
_archs = [a for a in ARCH_ORDER if a in df.arch.values]
_rows = {}
for arch in _archs:
    for config in ("s0", "s1"):
        r = _row(arch, {"platform": "fpga", "config": config}, "full")
        if r is None:
            continue
        st = sb[
            (sb.platform == "fpga")
            & (sb.model_name == r.model_name)
            & (sb.config == config)
            & (sb.scenario == "full")
        ]
        col = f"{ARCH_LABEL.get(arch, arch)} ({config})"
        for stg in STAGE_ORDER:
            row_st = st[st.stage == stg]
            if stg not in _rows:
                _rows[stg] = {}
            _rows[stg][col] = row_st.mean_ms.iloc[0] if not row_st.empty else float("nan")

# Column order: interleave s0/s1 per arch
_cols = [f"{ARCH_LABEL.get(a, a)} ({c})" for a in _archs for c in ("s0", "s1")]
_tbl = pd.DataFrame(_rows, index=_cols).T.reindex(STAGE_ORDER).dropna(how="all")
_tbl.index.name = "stage"
print(_tbl.to_string(float_format=lambda x: f"{x:6.2f}", na_rep="    --"))

if "normalize" in _tbl.index:
    _norm_mean = _tbl.loc["normalize"].mean()
    print(f"\nnormalize mean across all arch/config: {_norm_mean:.2f} ms")

# ── s0 → s1 saving for g_a and g_s ──────────────────────────────────────────
print("\ns0 → s1 saving (channel-parallel):")
print(f"  {'stage':<6}  {'arch':<6}  {'s0 [ms]':>8}  {'s1 [ms]':>8}  {'Δ [ms]':>8}  {'Δ [%]':>7}")
print(f"  {'-' * 6}  {'-' * 6}  {'-' * 8}  {'-' * 8}  {'-' * 8}  {'-' * 7}")
for stg in ("g_a", "g_s"):
    if stg not in _tbl.index:
        continue
    for arch in _archs:
        lbl = ARCH_LABEL.get(arch, arch)
        s0 = _tbl.loc[stg, f"{lbl} (s0)"]
        s1 = _tbl.loc[stg, f"{lbl} (s1)"]
        if pd.isna(s0) or pd.isna(s1):
            continue
        delta = s0 - s1
        pct = 100.0 * delta / s0
        print(f"  {stg:<6}  {lbl:<6}  {s0:>8.2f}  {s1:>8.2f}  {delta:>8.2f}  {pct:>6.1f}%")

## §2 · Energy per patch — FPGA only (4 arch)

Total energy/patch (mJ) on FPGA-s1 across all four architectures.

In [ ]:
def energy_bars_fpga(archs=None, config="s1", scenario="compress", save=None):
    """Energy/patch bars for FPGA only (4 archs)."""
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    spec = {"platform": "fpga", "config": config}
    vals = {a: _row(a, spec, scenario) for a in archs}
    vals = {a: r["energy_mJ_per_patch"] for a, r in vals.items() if r is not None}
    ymax = max(vals.values(), default=1)
    fig, ax = plt.subplots(figsize=(2.0 * len(archs) + 1.5, 4.5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.3, color="#aaa")
    for xi, arch in enumerate(archs):
        v = vals.get(arch)
        if v is None:
            continue
        ax.bar(xi, v, width=0.7, color=PLAT["fpga_dynamic"], edgecolor="white", lw=0.5, zorder=3)
        ax.text(xi, v + ymax * 0.012, f"{v:.1f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel("energy / patch (mJ)")
    ax.set_ylim(0, ymax * 1.18)
    fig.tight_layout()
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    plt.show()


energy_bars_fpga(save="fig_energy_fpga_4arch")

In [ ]:
# Reuse the full stage_breakdown + grouped_bars logic from benchmark_cross_platform_analysis.ipynb
# but restricted to ["FP", "ResSHyp"] and ["CPU", "GPU", "FPGA"].
ARCHS_3P = ["FP", "ResSHyp"]
SERIES_3P = ["CPU", "GPU", "FPGA"]


def grouped_bars_3p(
    metric, ylabel, archs=ARCHS_3P, series=SERIES_3P, scenario="compress", ref="CPU", save=None
):
    """Grouped bar chart — 3 platforms × 2 archs."""
    x = np.arange(len(archs))
    n = len(series)
    w = 0.8 / n
    vals = {
        (a, s): (
            _row(a, SERIES[s], scenario)[metric]
            if _row(a, SERIES[s], scenario) is not None
            and not pd.isna(_row(a, SERIES[s], scenario)[metric])
            else None
        )
        for a in archs
        for s in series
    }
    ymax = max((v for v in vals.values() if v is not None), default=1)
    fig, ax = plt.subplots(figsize=(3.0 * len(archs) + 1.5, 5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35, color="#aaa")
    for ai, arch in enumerate(archs):
        refv = vals.get((arch, ref))
        for si, s in enumerate(series):
            v = vals.get((arch, s))
            if v is None:
                continue
            xp = x[ai] + (si - (n - 1) / 2) * w
            spec = SERIES[s]
            ax.bar(xp, v, width=w, color=spec["color"], edgecolor="white", lw=0.5, zorder=3)
            ax.text(
                xp,
                v + ymax * 0.012,
                f"{v:.0f}" if v >= 10 else f"{v:.1f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
            if s != ref and refv:
                fac = (refv / v) if True else (v / refv)  # lower_is_better=True
                ax.text(
                    xp,
                    v + ymax * 0.065,
                    f"{fac:.0f}×" if fac >= 10 else f"{fac:.1f}×",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    fontweight="bold",
                    color="#222",
                )
    ax.set_xticks(x)
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, ymax * 1.22)
    ax.legend(
        handles=[
            Patch(facecolor=SERIES[s]["color"], edgecolor="white", label=SERIES[s]["label"])
            for s in series
        ],
        fontsize=10,
        framealpha=0.9,
    )
    fig.tight_layout()
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    plt.show()


grouped_bars_3p("total_latency_mean_ms", "latency / patch (ms)", save="fig_latency_3platform")
grouped_bars_3p("energy_mJ_per_patch", "energy / patch (mJ)", save="fig_energy_3platform")

## §3 · Cross-platform comparison table

Total latency [ms] and energy/patch [mJ] for CPU / GPU / FPGA across architectures and scenarios.

Call `cross_platform_table(archs=[...])` for any subset.  Pass `scenario="compress"` or
`scenario="full"` to drop the grouped header and show only one scenario.

In [ ]:
def cross_platform_table(
    archs=None,
    platforms=("CPU", "GPU", "FPGA"),
    scenario=None,
    ref="CPU",
    tabcolsep=6,
):
    """Print a booktabs LaTeX table: latency [ms] + energy/patch [mJ] per arch × platform,
    with ×ref improvement columns whose header is hidden under a multicolumn metric label.

    archs    : list of arch keys (default: ["FP", "ResSHyp"]).
    platforms: SERIES keys in display order (default: CPU → GPU → FPGA).
    scenario : None → both compress and full with a two-level grouped header
                      (\\multicolumn + \\cmidrule) inside a table* (two-column span);
               "compress" or "full" → single scenario, flat single-level header, table.
    ref      : SERIES key used as speedup baseline (default "CPU"); its rows show \\textemdash.
    tabcolsep: pt value for \\setlength{\\tabcolsep} (default 4 pt).
    """
    archs = archs or ["FP", "ResSHyp"]
    scenarios = [scenario] if scenario else ["compress", "full"]
    n_plat = len(platforms)
    n_scen = len(scenarios)

    # Each scenario block: S (lat) r (×CPU) S (energy) r (×CPU) — 4 cols per scenario
    scen_cols_spec = "S[table-format=6.1] l S[table-format=6.1] l"
    col_spec = "l l " + " ".join([scen_cols_spec] * n_scen)

    def _fmt_x(val, ref_val, is_ref):
        """Format a speedup/efficiency factor; \\textemdash for the baseline platform."""
        if is_ref:
            return r"\textemdash"
        if val is None or ref_val is None:
            return "{--}"
        ratio = ref_val / val
        return f"${ratio:.0f}\\times$" if ratio >= 10 else f"${ratio:.1f}\\times$"

    # Pre-compute reference values per (arch, scenario) — looked up once before the platform loop
    cpu_ref = {}
    for arch in archs:
        for sc in scenarios:
            r = _row(arch, SERIES[ref], sc)
            cpu_ref[(arch, sc)] = {
                "lat": float(r["total_latency_mean_ms"])
                if r is not None and pd.notna(r["total_latency_mean_ms"])
                else None,
                "nrg": float(r["energy_mJ_per_patch"])
                if r is not None and pd.notna(r["energy_mJ_per_patch"])
                else None,
            }

    # table* spans both columns in two-column layout; use it for the wider dual-scenario table
    tbl_env = "table*" if scenario is None else "table"

    lines = [
        rf"\begin{{{tbl_env}}}[t]",
        r"    \centering",
        r"    \caption{Latency and energy consumption per patch across different hardware platforms for the Compression scenario, $\times$ vs.\ CPU.}\label{tab:crossplatform_energy_latency}",
        rf"    \setlength{{\tabcolsep}}{{{tabcolsep}pt}}",
        rf"    \begin{{tabular}}{{{col_spec}}}",
        r"        \toprule",
    ]

    if scenario is None:
        # Row 1: grouped scenario names spanning 4 cols each
        scen_header = " & ".join(
            rf"\multicolumn{{4}}{{c}}{{{sc.capitalize()}}}" for sc in scenarios
        )
        lines.append(
            rf"        \multirow{{2}}{{*}}{{Arch.}} & \multirow{{2}}{{*}}{{Platform}}"
            rf" & {scen_header} \\"
        )
        # \cmidrule between the two header rows — one per scenario block
        lines.append(
            " ".join(rf"        \cmidrule(lr){{{3 + 4 * i}-{6 + 4 * i}}}" for i in range(n_scen))
        )
        # Row 2: metric labels as multicolumn{2} — ×ref column gets no explicit header
        sub = " & ".join(
            [r"\multicolumn{2}{c}{Latency\,[ms]} & \multicolumn{2}{c}{Energy\,[mJ]}"] * n_scen
        )
        lines.append(rf"         & & {sub} \\")
    else:
        # Single-level header: metric labels as multicolumn{2}, ×ref column hidden
        lines.append(
            r"        {Arch.} & {Platform}"
            r" & \multicolumn{2}{c}{Latency\,[ms]} & \multicolumn{2}{c}{Energy\,[mJ]} \\"
        )

    lines.append(r"        \midrule")

    for ai, arch in enumerate(archs):
        label = ARCH_LABEL.get(arch, arch)
        for pi, plat in enumerate(platforms):
            arch_cell = rf"\multirow{{{n_plat}}}{{*}}{{\texttt{{{label}}}}}" if pi == 0 else ""
            plat_label = SERIES[plat]["label"]
            is_ref = plat == ref

            data_cells = []
            for sc in scenarios:
                r = _row(arch, SERIES[plat], sc)
                ref_lat = cpu_ref[(arch, sc)]["lat"]
                ref_nrg = cpu_ref[(arch, sc)]["nrg"]

                lat_v = (
                    float(r["total_latency_mean_ms"])
                    if r is not None and pd.notna(r["total_latency_mean_ms"])
                    else None
                )
                nrg_v = (
                    float(r["energy_mJ_per_patch"])
                    if r is not None and pd.notna(r["energy_mJ_per_patch"])
                    else None
                )

                lat = f"{lat_v:.1f}" if lat_v is not None else "{--}"
                x_lat = _fmt_x(lat_v, ref_lat, is_ref)
                nrg = f"{nrg_v:.1f}" if nrg_v is not None else "{--}"
                x_nrg = _fmt_x(nrg_v, ref_nrg, is_ref)

                data_cells += [lat, x_lat, nrg, x_nrg]

            lines.append("        " + " & ".join([arch_cell, plat_label] + data_cells) + r" \\")

        if ai < len(archs) - 1:
            lines.append(r"        \midrule")

    lines += [r"        \bottomrule", r"    \end{tabular}", rf"\end{{{tbl_env}}}"]
    print("\n".join(lines))


cross_platform_table(archs=ARCH_ORDER, scenario="compress")

## §4 · Network statistics

Per-subnet and per-architecture parameter counts, FP32 memory footprint, operation count,
and maximum DPU throughput — derived from `xdputil` static analysis (`_roofline/*_xmodel_info.json`).

- **#Params** = `const_bytes` (INT8 weight count; DPU deploys INT8)
- **Mem [MB]** = `const_bytes × 4 / 1e6` (FP32 model footprint; DPU ≈ 1/4 of this)
- **#OPs [G]** = `workload_ops / 1e9` (per-subnet; single forward pass)
- **Max DPU FPS** = `peak_fps` (single-image, single-DPU; s0 config)

**Table 2 — inference scenario costs** apply a **×2 MERLIN factor**: MERLIN processes the real and
imaginary SAR components as two separate inputs, so every DPU subgraph executes twice per patch.
Scenario subnets:
- **Compress**: g_a [+ h_a for SH/ResSH]
- **Full** (compress + decompress): g_a + h_a + h_s + g_s [FP/ResFP: g_a + g_s only]

Matches `\label{tab:network_stats}` in the manuscript.

In [ ]:
import json
from pathlib import Path

ROOFLINE_DIR = ROOT / "results" / "benchmark_hardware" / "_roofline"

_XMODEL_FILES = {
    "FP": "FP-relu_s0_L1000_pt_xmodel_info.json",
    "SHyp": "SHyp-relu_s0_L1000_pt_xmodel_info.json",
    "ResFP": "ResFP-relu_s0_L1000_pt_xmodel_info.json",
    "ResSHyp": "ResSHyp-relu_s0_L1000_pt_xmodel_info.json",
}

# Subnets active per scenario. MERLIN executes each subnet ×2 (real + imag).
_COMPRESS_SUBNETS = {
    "FP": ["g_a"],
    "SHyp": ["g_a", "h_a"],
    "ResFP": ["g_a"],
    "ResSHyp": ["g_a", "h_a"],
}
_FULL_SUBNETS = {
    "FP": ["g_a", "g_s"],
    "SHyp": ["g_a", "h_a", "h_s", "g_s"],
    "ResFP": ["g_a", "g_s"],
    "ResSHyp": ["g_a", "h_a", "h_s", "g_s"],
}


def load_xmodel_stats(roofline_dir: Path, xmodel_files: dict) -> dict:
    """Return {arch: {subgraph: {params_M, mem_mb, ops_g, fps}} + '_total' with scenario OPs."""
    out = {}
    for arch, fname in xmodel_files.items():
        p = roofline_dir / fname
        if not p.exists():
            print(f"WARNING: {p} not found — skipping {arch}")
            continue
        d = json.loads(p.read_text())
        sgs = {}
        for sg, v in d["subgraphs"].items():
            sgs[sg] = dict(
                params_M=v["const_bytes"] / 1e6,
                mem_mb=v["const_bytes"] * 4 / 1e6,
                ops_g=v["workload_ops"] / 1e9,
                fps=v["peak_fps"],
            )
        total_bytes = sum(v["const_bytes"] for v in d["subgraphs"].values())
        # ×2: MERLIN processes real + imaginary SAR components separately
        ops_compress = 2.0 * sum(
            d["subgraphs"][sg]["workload_ops"]
            for sg in _COMPRESS_SUBNETS.get(arch, [])
            if sg in d["subgraphs"]
        )
        ops_full = 2.0 * sum(
            d["subgraphs"][sg]["workload_ops"]
            for sg in _FULL_SUBNETS.get(arch, [])
            if sg in d["subgraphs"]
        )
        sgs["_total"] = dict(
            params_M=total_bytes / 1e6,
            mem_mb=total_bytes * 4 / 1e6,
            ops_compress_g=ops_compress / 1e9,
            ops_full_g=ops_full / 1e9,
        )
        out[arch] = sgs
    return out


xstats = load_xmodel_stats(ROOFLINE_DIR, _XMODEL_FILES)

# ── Subnet display order (matches LaTeX table grouping) ──────────────────────
_SUBNET_ROWS = [
    ("g_a+Res", ["ResFP", "ResSHyp"], "g_a"),
    ("g_s+Res", ["ResFP", "ResSHyp"], "g_s"),
    ("g_a", ["FP", "SHyp"], "g_a"),
    ("g_s", ["FP", "SHyp"], "g_s"),
    ("h_a", ["SHyp", "ResSHyp"], "h_a"),
    ("h_s", ["SHyp", "ResSHyp"], "h_s"),
]

# ── Table 1: per-subnet (text) ────────────────────────────────────────────────
print("Table 1 — Per-subnet network statistics  (s0 xmodel, L=1000 weights)\n")
print(
    f"{'Sub-net':<12}  {'Architectures':<22}  {'#Params':>10}  {'Mem [MB]':>9}  {'#OPs [G]':>10}  {'Max FPS':>9}"
)
print("-" * 82)
for disp, archs, sg_key in _SUBNET_ROWS:
    rows = [xstats[a][sg_key] for a in archs if a in xstats and sg_key in xstats[a]]
    if not rows:
        continue
    r = rows[0]
    arch_str = " / ".join(ARCH_LABEL.get(a, a) for a in archs)
    print(
        f"{disp:<12}  {arch_str:<22}  "
        f"{r['params_M']:>9.2f}M  {r['mem_mb']:>9.2f}  "
        f"{r['ops_g']:>10.3f}  {r['fps']:>9.1f}"
    )

# ── Table 2: per-architecture inference cost (text) ───────────────────────────
print(
    "\n\nTable 2 — Per-architecture inference cost"
    "  (×2 MERLIN factor: real + imaginary processed separately)\n"
)
print(
    f"{'Arch.':<8}  {'#Subnets':>9}  {'#Params [M]':>12}  {'Mem [MB]':>9}  "
    f"{'#OPs full [G]':>14}  {'#OPs comp. [G]':>15}"
)
print("-" * 78)
for arch in ARCH_ORDER:
    if arch not in xstats:
        continue
    t = xstats[arch]["_total"]
    n_sg = len(xstats[arch]) - 1
    print(
        f"{ARCH_LABEL.get(arch, arch):<8}  {n_sg:>9}  "
        f"{t['params_M']:>11.2f}  {t['mem_mb']:>9.2f}  "
        f"{t['ops_full_g']:>14.2f}  {t['ops_compress_g']:>15.2f}"
    )

# ── LaTeX — tab:subnetwork_stats ──────────────────────────────────────────────
print("\n\n% LaTeX — tab:subnetwork_stats\n")
_rows_tex = [
    r"\begingroup",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{l l S[table-format=2.3] S[table-format=2.2] S[table-format=2.3] S[table-format=5.1]}",
    r"\toprule",
    r"{Sub-net} & {Architectures} & {\#Params\,[M]} & {Mem.\,[MB]} & {\#OPs\,[G]} & {Max DPU FPS} \\",
    r"\midrule",
]
for disp, archs, sg_key in _SUBNET_ROWS:
    rows = [xstats[a][sg_key] for a in archs if a in xstats and sg_key in xstats[a]]
    if not rows:
        continue
    r = rows[0]
    arch_str = " / ".join(r"\texttt{" + ARCH_LABEL.get(a, a) + "}" for a in archs)
    disp_tex = disp.replace("+Res", r"+\text{Res}")
    _rows_tex.append(
        f"${disp_tex}$ & {arch_str} & "
        f"{r['params_M']:.3f} & {r['mem_mb']:.2f} & {r['ops_g']:.3f} & {r['fps']:.1f} \\\\"
    )
_rows_tex += [r"\bottomrule", r"\end{tabular}", r"\endgroup"]
print("\n".join(_rows_tex))

# ── LaTeX — tab:network_stats ─────────────────────────────────────────────────
# Columns: Arch. | #Subnets | #Params [M] | Mem [MB] | #OPs full [G] | #OPs comp. [G]
# full before compress; \texttt{} for arch names; S columns for all numerics except #Subnets.
print("\n\n% LaTeX — tab:network_stats\n")
_rows_arch = [
    r"\begingroup",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{l r S[table-format=2.2] S[table-format=2.2] S[table-format=3.2] S[table-format=2.2]}",
    r"\toprule",
    r"{Arch.} & {\#Subnets} & {\#Params\,[M]} & {Mem.\,[MB]} & {\#OPs\,full\,[G]} & {\#OPs\,comp.\,[G]} \\",
    r"\midrule",
]
for arch in ARCH_ORDER:
    if arch not in xstats:
        continue
    t = xstats[arch]["_total"]
    n_sg = len(xstats[arch]) - 1
    arch_label = ARCH_LABEL.get(arch, arch)
    _rows_arch.append(
        f"\\texttt{{{arch_label}}} & {n_sg} & "
        f"{t['params_M']:.2f} & {t['mem_mb']:.2f} & "
        f"{t['ops_full_g']:.2f} & {t['ops_compress_g']:.2f} \\\\"
    )
_rows_arch += [r"\bottomrule", r"\end{tabular}", r"\endgroup"]
print("\n".join(_rows_arch))